# 03 · Exploratory Data Analysis

**Input:** `gold_train.parquet`, `gold_test.parquet`  
**Output:** `figures/eda_*.pdf`, `reports/eda_summary.csv`

In [ ]:
# Setup
from pathlib import Path
import pandas as pd, numpy as np
from scipy import stats
import plotly.graph_objects as go
import plotly.express as px

GOLD_DIR   = Path('../data/gold')
REPORT_DIR = Path('../reports'); REPORT_DIR.mkdir(parents=True, exist_ok=True)

PRIMARY, SECONDARY, GREY = '#075020', '#8A7A2F', "#969490"

PLOT_WIDTH = 1000
PLOT_HEIGHT = 650

## Feature Relationships — Correlation Analysis

Examine multivariate feature dependencies to identify redundancy and collinearity.

In [161]:
# Load & compute correlation

df = pd.read_parquet(GOLD_DIR / 'gold_train.parquet')

numeric_features = ['rolling_form_3', 'rolling_form_5', 'rolling_form_10',
                     'rolling_margin_3', 'h2h_winrate', 'consecutive_wins',
                     'experience', 'experience_diff', 'elo_diff_pre',
                     'wr_rank_diff', 'rwc_cumul_score', 'win']

corr_data = df[[c for c in numeric_features if c in df.columns]].dropna()
corr_matrix = corr_data.corr()

print(f'Features: {len(corr_matrix)} | Complete cases: {len(corr_data):,}')

# Correlation Heatmap

fig = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.columns,
    colorscale=[[0, SECONDARY], [1, PRIMARY]],
    colorbar=dict(title='Pearson r', thickness=15, len=0.7),
    hovertemplate='%{x} ↔ %{y}<br>r = %{z:.3f}<extra></extra>',
    text=corr_matrix.values,
    texttemplate='%{text:.2f}',
    textfont=dict(size=9)
))

fig.update_layout(
    template='plotly_white',
    width=PLOT_WIDTH,
    height=PLOT_HEIGHT,
    font=dict(size=10),
    xaxis_tickangle=-45,
    margin=dict(l=50, r=50, t=20, b=50)
)

fig.show()

Features: 12 | Complete cases: 2,044


In [165]:
df    = (pd.concat([pd.read_parquet(GOLD_DIR/f)
                    for f in ('gold_train.parquet','gold_test.parquet')])
           .sort_values('date').reset_index(drop=True))
df_sa = df[df.team == 'South Africa'].copy()
print(f'All: {len(df):,} | SA: {len(df_sa)} | SA win rate: {df_sa.win.mean():.1%}')

All: 3,410 | SA: 332 | SA win rate: 62.3%


In [174]:
yr = df_sa.groupby('year').win.agg(rate='mean', n='count')
yr['trend'] = yr.rate.rolling(3, center=True, min_periods=1).mean()
mean_wr = df_sa.win.mean()

fig = go.Figure()

# Bar chart
fig.add_trace(go.Bar(
    x=yr.index,
    y=yr.rate,
    marker=dict(color=PRIMARY),
    name='Annual',
    hovertemplate='Win rate: %{y:.1%}<extra></extra>',
    showlegend=True
))

# Line: 3-year trend
fig.add_trace(go.Scatter(
    x=yr.index,
    y=yr['trend'],
    mode='lines',
    name='3-year trend',
    line=dict(color=SECONDARY, width=3),
    hovertemplate='Trend: %{y:.1%}<extra></extra>'
))

# Mean line
fig.add_hline(
    y=mean_wr,
    line_dash='dot',
    line_color=GREY,
    annotation_text=f'Mean {mean_wr:.1%}',
    annotation_position='right'
)

fig.update_layout(
    xaxis_title='Year',
    yaxis_title='Win Rate',
    template='plotly_white',
    width=PLOT_WIDTH,
    height=PLOT_HEIGHT,
    yaxis=dict(tickformat='.0%', range=[0, 1.0]),
    xaxis=dict(range=[1994, 2025]),
    hovermode='x unified',
    legend=dict(x=0.95, y=0.98, xanchor='left', yanchor='top'),
    margin=dict(l=20, r=20, t=20, b=20)
)

fig.show()

In [191]:
loss_draw_margin = df_sa[df_sa.win==0].score_margin.dropna()
win_margin = df_sa[df_sa.win==1].score_margin.dropna()

fig = go.Figure()

fig.add_trace(go.Histogram(
    x=win_margin,
    name='Win',
    marker=dict(color=PRIMARY),
    nbinsx=30,
    opacity=0.95,
    hovertemplate='Score Margin: %{x:.2f}<br>Frequency: %{y}<extra></extra>'
))

fig.add_trace(go.Histogram(
    x=loss_draw_margin,
    name='Loss/Draw',
    marker=dict(color=SECONDARY),
    nbinsx=30,
    opacity=0.95,
    hovertemplate='Score Margin: %{x:.2f}<br>Frequency: %{y}<extra></extra>'
))

fig.add_vline(x=0, line_dash='solid', line_color='black', line_width=1)

fig.update_layout(
    xaxis_title='Score Margin',
    yaxis_title='Frequency',
    barmode='overlay',
    template='plotly_white',
    width=PLOT_WIDTH,
    height=PLOT_HEIGHT,
    hovermode='closest',
    legend=dict(x=0.9, y=1.1, xanchor='left', yanchor='top'),
    margin=dict(l=50, r=50, t=50, b=50)
)

fig.show()

In [186]:
loss_draw = df_sa[df_sa.win==0].rolling_form_3.dropna()
win = df_sa[df_sa.win==1].rolling_form_3.dropna()

bins = np.linspace(0, 1.1, 21)
hist_loss, edges = np.histogram(loss_draw, bins=bins)
hist_win, _ = np.histogram(win, bins=bins)
bin_centers = (edges[:-1] + edges[1:]) / 2

fig = go.Figure()

fig.add_trace(go.Bar(
    x=bin_centers - 0.035,
    y=hist_loss,
    width=0.07,
    name='Loss/Draw',
    marker=dict(color=SECONDARY),
    hovertemplate='Rolling form: %{x:.2f}<br>Count: %{y}<extra></extra>'  # ← Mehr Info!
))

fig.add_trace(go.Bar(
    x=bin_centers + 0.035,
    y=hist_win,
    width=0.07,
    name='Win',
    marker=dict(color=PRIMARY),
    hovertemplate='Rolling form: %{x:.2f}<br>Count: %{y}<extra></extra>'  # ← Mehr Info!
))

fig.update_layout(
    xaxis_title='Win Rate Prior 3 Matches',
    yaxis_title='Frequency',
    barmode='group',
    template='plotly_white',
    height=PLOT_HEIGHT,
    width=PLOT_WIDTH,
    hovermode='closest',
    legend=dict(x=1.1, y=1.1, xanchor='right', yanchor='top'),
    margin=dict(l=60, r=100, t=60, b=60)
)

fig.show()

In [195]:
ha = df_sa.groupby('home').win.agg(['mean','count']).reset_index()
ha['venue'] = ha['home'].map({0: 'Away', 1: 'Home'})

contingency = pd.crosstab(df_sa['home'], df_sa['win'])
chi2, p_value = stats.chi2_contingency(contingency)[:2]

sig_text = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else 'ns'
diff = ha.iloc[1]['mean'] - ha.iloc[0]['mean']

fig = go.Figure()

fig.add_trace(go.Bar(
    x=ha['venue'],
    y=ha['mean'],
    marker=dict(color=[SECONDARY, PRIMARY]),
    text=[f"{row['mean']:.1%}<br>(n={int(row['count'])})" 
          for _, row in ha.iterrows()],
    textposition='outside',
    hovertemplate='%{x}<br>Win rate: %{y:.1%}<extra></extra>',
    showlegend=False
))

fig.update_layout(
    xaxis_title='Venue',
    yaxis_title='Win Rate',
    template='plotly_white',
    height=650,
    width=1000,
    yaxis=dict(tickformat='.0%', range=[0, 1.0]),
    margin=dict(l=60, r=60, t=60, b=60),
        annotations=[
        dict(
            x=1.05, y=1.05,
            text=f'χ² = {chi2:.1f}, p < 0.001 {sig_text}<br>Difference: +{diff:.1%}',
            showarrow=False,
            font=dict(size=11),
            xref='paper', yref='paper',
            bgcolor='rgba(255,255,255,0.8)',
            borderwidth=1,
            xanchor='right',
            yanchor='top'
        )
    ]
)

fig.show()
print(f"Chi-squared test: χ² = {chi2:.2f}, p = {p_value:.4f}")

Chi-squared test: χ² = 23.17, p = 0.0000


In [202]:
loss_elo = df[df.win==0].elo_diff_pre.dropna()
win_elo = df[df.win==1].elo_diff_pre.dropna()

fig = go.Figure()

fig.add_trace(go.Histogram(
    x=win_elo,
    name='Win',
    marker=dict(color=PRIMARY),
    nbinsx=40,
    opacity=1,
    hovertemplate='Elo diff: %{x:.2f}<br>Frequency: %{y}<extra></extra>'
))

fig.add_trace(go.Histogram(
    x=loss_elo,
    name='Loss/Draw',
    marker=dict(color=SECONDARY),
    nbinsx=40,
    opacity=0.9,
    hovertemplate='Elo diff: %{x:.2f}<br>Frequency: %{y}<extra></extra>'
))

fig.add_vline(x=0, line_dash='solid', line_color='black', line_width=1)

fig.update_layout(
    xaxis_title='Elo Difference (Team − Opponent)',
    yaxis_title='Frequency',
    barmode='overlay',
    template='plotly_white',
    height=PLOT_HEIGHT,
    width=PLOT_WIDTH,
    hovermode='closest',
    legend=dict(x=1.02, y=1.1, xanchor='right', yanchor='top'),
    margin=dict(l=60, r=60, t=60, b=60)
)

fig.show()

In [211]:
team_stats = (df.groupby('team')
                .agg(win_rate=('win','mean'), matches=('win','count'),
                     rwc_score=('rwc_cumul_score','mean'))
                .query('matches >= 20').reset_index())

sa_stats = team_stats[team_stats['team'] == 'South Africa']
other_stats = team_stats[team_stats['team'] != 'South Africa']

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=other_stats['rwc_score'],
    y=other_stats['win_rate'],
    mode='markers+text',
    marker=dict(
        size=np.sqrt(other_stats['matches']) * 2.5,
        color=other_stats['win_rate'],
        colorscale=[[0, SECONDARY], [1, PRIMARY]],
        showscale=True,
        colorbar=dict(title='Win Rate'),
        line=dict(color='black', width=0.5),
        opacity=1.0
    ),
    text=other_stats['team'],
    textposition='top center',
    textfont=dict(size=8),
    hovertemplate='<b>%{text}</b><br>RWC pedigree: %{x:.2f}<br>Win rate: %{y:.1%}<br>Matches: %{customdata}<extra></extra>',
    customdata=other_stats['matches'],
    showlegend=False
))

# RSA
fig.add_trace(go.Scatter(
    x=sa_stats['rwc_score'],
    y=sa_stats['win_rate'],
    mode='markers+text',
    marker=dict(
        size=np.sqrt(sa_stats['matches']) * 2.5,
        color=sa_stats['win_rate'],
        colorscale=[[0, SECONDARY], [1, PRIMARY]],
        line=dict(color='black', width=2),
        opacity=1.0
    ),
    text=sa_stats['team'],
    textposition='top center',
    textfont=dict(size=12, color='black'),
    hovertemplate='<b>%{text}</b><br>RWC pedigree: %{x:.2f}<br>Win rate: %{y:.1%}<br>Matches: %{customdata}<extra></extra>',
    customdata=sa_stats['matches'],
    showlegend=False
))

fig.update_layout(
    xaxis_title='Mean RWC Cumulative Score',
    yaxis_title='Win Rate',
    template='plotly_white',
    height=PLOT_HEIGHT,
    width=PLOT_WIDTH,
    yaxis=dict(tickformat='.0%'),
    hovermode='closest',
    margin=dict(l=60, r=100, t=60, b=60)
)

fig.show()

In [212]:
stats = (df.groupby('team')
           .agg(matches=('win','count'), win_rate=('win','mean'),
                avg_margin=('score_margin','mean'),
                avg_elo_diff=('elo_diff_pre','mean'),
                rwc_pedigree=('rwc_cumul_score','mean'))
           .round(3).sort_values('win_rate', ascending=False))
stats.to_csv(REPORT_DIR / 'eda_summary.csv')
print('Saved: eda_summary.csv')
print(stats.to_string())

Saved: eda_summary.csv
              matches  win_rate  avg_margin  avg_elo_diff  rwc_pedigree
team                                                                   
New Zealand       343     0.808      18.437       276.666        18.029
South Africa      332     0.623       7.702        91.118        10.711
England           329     0.617       7.043        94.206        14.234
France            324     0.571       4.009        43.427        15.318
Ireland           290     0.562       3.793        25.310         8.883
Australia         354     0.559       4.282        59.039        17.506
Wales             317     0.429      -1.861       -54.741         9.230
Samoa              36     0.389      -3.639       -33.096         6.028
Scotland          277     0.386      -3.303      -102.553        11.271
Fiji               37     0.351      -6.486       -82.240         5.081
Hong Kong           3     0.333     -22.000        83.324         0.000
Argentina         245     0.322      -6.6

In [213]:
silver = pd.read_parquet('../data/silver/silver_results.parquet')
print(silver.groupby('team').size().sort_values().to_string())

team
Kenya             3
Spain             3
Ivory Coast       3
Hong Kong         3
Chile             4
Zimbabwe          6
Russia            8
Portugal         11
Uruguay          19
Georgia          24
Namibia          26
USA              32
Canada           32
Romania          32
Tonga            33
Samoa            36
Japan            37
Fiji             37
Argentina       245
Italy           250
Scotland        277
Ireland         290
Wales           317
France          324
England         329
South Africa    332
New Zealand     343
Australia       354
